# 第2回 Pythonによる金融データの操作

2024年のドル・円レート（日銀、東京17時）を使って、ドルの支払い額、円からドルへの両替、ドル保有の損益を計算する。Pythonの基本操作も、この計算の中で練習する。演習は3題ある。

## 実行の準備

`finance_data.zip` をNotebookと同じフォルダに置き、下のセルを実行する。セルは上から順に実行すること。「変数が見つからない」というエラーが出たら、前のセルを実行し忘れている。

In [ ]:
from pathlib import Path
from zipfile import ZipFile

if not Path("course_data.py").exists():
    with ZipFile("finance_data.zip") as archive:
        archive.extractall(".")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from course_data import (
    load_boj_fx, load_boj_monthly, load_fed_fx, load_fed_rates,
    configure_plotting, format_axis,
)

configure_plotting()
pd.options.display.float_format = "{:.3f}".format


## 表の読み込み

日付つきのデータは pandas の表（DataFrame）で扱う。`load_boj_fx()` で日銀のレートを読み込み、結果に `fx` という名前を付ける。`=` は右の結果に左の名前を付ける操作である。

In [ ]:
fx = load_boj_fx()
fx.head(8)


`date` が日付、`usd_jpy` が円／ドルのレート。左端の日付は各行の見出しで、索引（インデックス）と呼ぶ。`NaN` は欠測で、その日の値が記録されていないことを示す。休日など、市場が閉まっている日がこれに当たる。

In [ ]:
fx.tail()


## 列の選択と欠測

`fx["usd_jpy"]` でレートの列を取り出し、`.dropna()` で欠測の日を除く。列を1本取り出したものはSeries（日付と値が対応した1列のデータ）と呼ぶ。この結果を `rate` とする。

In [ ]:
rate = fx["usd_jpy"].dropna()
rate.head()


## 日付で期間を選ぶ

日付の索引を持つSeriesは、`.loc["2024"]` で2024年、`.loc["2024-01":"2024-03"]` で1〜3月を取り出せる。位置で選ぶ `.iloc` に対して `.loc` は日付などのラベルで選ぶ。ここでは、2024年の系列を `rate_2024` とする。欠測を除いているため、観測数は暦日数と一致しない。

In [ ]:
rate_2024 = rate.loc["2024"]
print(f"2024年の観測数：{len(rate_2024):,}")
rate.loc["2024-01":"2024-03"].head()


## 変数と計算

2024年の最初と最後のレートを `first` と `last` に入れ、差を `change` とする。`.iloc[0]` が先頭、`.iloc[-1]` が末尾。年初と年末は休日のことがあるので、日付も表示する。

In [ ]:
first = rate_2024.iloc[0]
last = rate_2024.iloc[-1]
change = last - first

print(f"最初の観測：{rate_2024.index[0].date()}、{first:.2f} 円／ドル")
print(f"最後の観測：{rate_2024.index[-1].date()}、{last:.2f} 円／ドル")
print(f"変化額：{change:.2f} 円／ドル")


`change` が正なら円安・ドル高、負なら円高・ドル安。

`print()` は結果を表示する。`{change:.2f}` は小数第2位までの表示で、変数の値は変わらない。

## ドル建て金額と円の予算

1ドル150円のとき、10ドルは 10 × 150 ＝ 1,500円になる。レートが160円なら、同じ10ドルが 10 × 160 ＝ 1,600円になる。

文字で書く。レートを $S$ 円／ドル、金額を $Q$ ドルとして、円での金額 $C$ は

$$C=Q S$$

単位は「ドル × 円／ドル ＝ 円」。受け取ったドルを円に換えるときも同じ式になる。

逆に、1,500円で買えるドルは 1,500 ÷ 150 ＝ 10ドル。レートが160円なら 1,500 ÷ 160 ＝ 9.375 で、約9.4ドルになる。

円の予算 $B$ で買えるドルは

$$Q=\frac{B}{S}$$

同じ予算でも、レートが高いほど買えるドルは少なくなる。

金額は計算用の仮の値である。手数料は考えず、観測レートでそのまま交換できるものとする。

In [ ]:
usd_amount = 1_000
yen_budget = 100_000

print(f"年初のレートで1,000ドルを円換算：{usd_amount * first:,.0f} 円")
print(f"年初のレートで10万円から買えるドル：{yen_budget / first:,.2f} ドル")


`1_000` は `1000` と同じ。`_` は桁の区切りである。`:,.0f` は小数なしのカンマ区切り、`:,.2f` は小数第2位まで。

`#` から後ろはコメントであり、実行されない。式を書いたら `Shift + Enter`。記入欄はダブルクリックで編集する。

## 演習1：ドル建て支払いと円安

2,000ドルの同じ支払いを、2024年の最初の観測日と最後の観測日にそれぞれ円で用意する場合を比較する。為替レートを `first` と `last` から取り出し、金額を計算せよ。

1. それぞれ何円必要か。最後の日の支払い額から最初の日の支払い額を引くと何円か。
2. この変化は円安・円高のどちらか。ドル建て金額が同じなのに円の支払い額が変わる理由を説明せよ。
3. 支払いの代わりに2,000ドルを受け取り、その日に円に換える人なら、円での受取額はどう変わるか。「円安は全員にとって損である」といえるか。

計算の前に増減を予想する。

In [ ]:
payment_usd_ex1 = 2_000
# 最初の日と最後の日の円換算額、および差を計算する。
# print() を使い、円の単位も付けて表示する。


### 演習1の記入欄

- 最初の日／最後の日の支払い額と差：
- 円安・円高の判断と理由：
- ドルを受け取る人への影響：


<details>
<summary>演習1のヒント</summary>

`payment_usd_ex1 * first` と `payment_usd_ex1 * last` で円換算額を求める。差は「最後 − 最初」。同じ増加額でも、支払う人には支出増、受け取る人には収入増になる。

</details>

## データ型と比較

数値と文字列は別の型で、文字列の `"ドル・円"` は割り算できない。型は `type()` で調べる。`last > first` のような比較は `True` か `False` を返す。

In [ ]:
label = "ドル・円"
print(type(label))
print(type(first))
print(last > first)


`if` で処理を分ける。`change` が正ならこの処理、負なら `elif` の処理、どちらでもなければ `else` の処理。`:` と字下げが処理の範囲を示す。

In [ ]:
if change > 0:
    print("2024年の最初と最後の観測を比べると、円安・ドル高です。")
elif change < 0:
    print("2024年の最初と最後の観測を比べると、円高・ドル安です。")
else:
    print("2024年の最初と最後の観測値は同じです。")


## リストと繰り返し

値を並べたものがリスト。先頭3件を `.tolist()` でリストにする。`.iloc[:3]` は先頭から3件、`len()` は件数、`values[0]` は先頭の要素。日付の情報は消える。

In [ ]:
values = rate_2024.iloc[:3].tolist()
print(values)
print(len(values))
print(values[0])


`for value in values:` は要素を1つずつ `value` に入れ、字下げした処理を繰り返す。`.iloc[:3]` を `.iloc[:5]` に変えて、2つのセルを実行し直してみよう。

In [ ]:
for value in values:
    print(f"1ドル = {value:.2f} 円なら、1,000ドルの支払いは {1_000 * value:,.0f} 円")


## レートの推移

折れ線にすると、水準が高い時期や大きく動いた時期が見える。`fig` は図全体、`ax` は描く領域。`ax.plot()` で線を描き、`ax.set()` で題名と単位を付ける。`format_axis()` は目盛を整える関数で、値は変えない。

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(rate_2024.index, rate_2024, linewidth=1)
ax.set(title="東京市場のドル・円（2024年）", ylabel="円／ドル")
format_axis(ax)
fig.tight_layout()
plt.show()


## 期間の代表値

`.mean()`、`.min()`、`.max()` で平均・最小・最大を求める。平均は観測のある各日を同じ重みで平均したもの。

In [ ]:
print(f"平均：{rate_2024.mean():.2f} 円／ドル")
print(f"最小：{rate_2024.min():.2f} 円／ドル")
print(f"最大：{rate_2024.max():.2f} 円／ドル")


## 円の予算と購入できるドル

最小・最大の日付は `.idxmin()`、`.idxmax()` で分かる。

たとえば15,000円で100ドル以上を買いたいとする。上限は 15,000 ÷ 100 ＝ 150円／ドルで、150円以下の日が条件に合う。

文字で書くと、予算 $B$ で $Q^*$ ドル以上を買えるのは、レートが上限以下の日である（$B,Q^*,S>0$）。

$$\frac{B}{S}\geq Q^* \quad\Longleftrightarrow\quad S\leq\frac{B}{Q^*}$$

`rate_2024 <= limit` を `.loc[...]` に入れると、その日だけを選べる。

## 演習2：12万円の両替予算

2024年の各観測日に、12万円を一度にドルへ換えるとする。

1. 最も多くのドルを買える日と最も少なくなる日を求め、それぞれの日付・為替レート・購入できるドルを示せ。
2. 800ドルを買うために、円／ドルのレートはいくら以下である必要があるか。予算と目標額から計算せよ。

追加問題：その条件に合う観測を選び、日数を数えよ。全245観測日のうち何%かも求め、この割合を「翌年に800ドルを買える確率」と呼べるか考えよ。

休日・欠測日のレートは補わない。

In [ ]:
budget_ex2 = 120_000
goal_usd_ex2 = 800
# rate_2024.min(), .max(), .idxmin(), .idxmax() を使う。
# 予算内で800ドルを買えるレートの上限を計算する。
# 条件に合う観測を .loc[...] で選び、len(...) で数える。


### 演習2の記入欄

- 最も多く買える日・レート・ドルの金額：
- 最も少なくなる日・レート・ドルの金額：
- 800ドルを買えるレートの上限：
- 追加問題：条件に合う日数・割合と、翌年について判断するために不足していること：


<details>
<summary>演習2のヒント</summary>

レートが低い日ほど多く買える。`budget_ex2 / rate_2024.min()` のように計算する。条件に合う観測は `limit_ex2 = budget_ex2 / goal_usd_ex2` として `rate_2024.loc[rate_2024 <= limit_ex2]` で選ぶ。割合を将来の確率とみなすには、将来の為替の動きについての仮定が要る。

</details>

## ドル保有の円換算額と損益

10万円を2024年の最初の観測日にドルへ換え、年末まで持ち続ける。途中の売買、利息、手数料、税金は考えない。

たとえば10万円を1ドル150円でドルに換えると、100,000 ÷ 150 ≒ 666.7ドルになる。レートが160円になれば、666.7 × 160 ≒ 106,667円で、6,667円増える。

同じ計算を文字で書く。最初の円資金を $B_0$、そのときのレートを $S_0$ とすると買えるドルは $Q=B_0/S_0$。日 $t$ のレートを $S_t$ とすると、円換算額は

$$V_t=Q S_t=B_0\frac{S_t}{S_0}$$

ドルの量が同じでも円換算額は毎日動く。円に戻していない間の値は、その日のレートでの評価額である。

損益は $V_t-B_0$。円建て収益率は

$$R_t=\frac{V_t-B_0}{B_0}=\frac{S_t}{S_0}-1$$

$R_t=0.01$ は1%。%で表すには100を掛ける。上の例なら $160/150-1\approx 0.067$ で、6.7%の増加になる。

In [ ]:
initial_yen = 100_000
usd_held = initial_yen / rate_2024.iloc[0]
yen_value = usd_held * rate_2024

print(f"最初の観測日：{rate_2024.index[0].date()}")
print(f"保有するドル：{usd_held:,.2f} ドル")
for day in yen_value.index[:3]:
    print(f"{day.date()} の円換算額：{yen_value.loc[day]:,.0f} 円")


In [ ]:
from matplotlib.ticker import StrMethodFormatter

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(yen_value.index, yen_value, label="保有ドルの円換算額", linewidth=1)
ax.axhline(initial_yen, color="gray", linestyle="--", label="最初の円資金：10万円")
ax.set(title="年初に購入したドルを保有した場合（2024年）", ylabel="円")
format_axis(ax)
ax.yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
ax.legend()
fig.tight_layout()
plt.show()


破線（10万円）より下の日は円換算額が最初の資金を下回っている。このように、為替の変動で円換算額や支払い額が変わる不確実性を為替リスクという。

## 演習3：年末の利益と途中の損失

1. 年末の円換算額、最初の10万円との差、円建て収益率（%）を計算せよ。
2. 保有中の円換算額が最も小さかった日と金額を求め、最初の円資金を下回ったか確認せよ。「年末に利益が出たので、途中で損失になる心配はなかった」という説明の問題を述べよ。

追加問題：年初の代わりに、7月の最初の観測日に10万円をドルへ換え、年末まで保有した場合の損益を計算せよ。ドルの購入量から計算し直し、年初に購入した場合と比べよ。

追加問題では `rate_2024.loc["2024-07":"2024-12"]` を使う。資金は同じ10万円で、買えるドルの量は購入レートで変わる。

In [ ]:
# yen_value.iloc[-1] から年末の円換算額を取り出す。
# 最初の円資金との差と、その資金に対する割合を求める。
# yen_value.min() と .idxmin() で保有中の最小値と日付を求める。

july_december_ex3 = rate_2024.loc["2024-07":"2024-12"]
# 7月からの比較では、10万円で購入できるドルを計算し直す。


### 演習3の記入欄

- 年初から保有した場合の年末評価額・損益・収益率：
- 最小の円換算額・日付と、途中の損失についての説明：
- 追加問題：7月から保有した場合の損益と、期間の選択について分かったこと：


<details>
<summary>演習3のヒント</summary>

年末の円換算額を `end_value = yen_value.iloc[-1]` として取り出す。損益は `end_value - initial_yen`、収益率はその損益を `initial_yen` で割る。%表示では100を掛ける。

保有中の最小値は `yen_value.min()`、その日付は `yen_value.idxmin()` で求める。追加問題では `initial_yen / july_december_ex3.iloc[0]` で7月に買えるドルを求め、その値に12月最後のレートを掛ける。

</details>


## 発展：表示方向と変化率

ドル／円のレートは円／ドルのレート $S$ の逆数 $1/S$ である。

1. 2024年の最初と最後のレートを逆数にし、それぞれの表示方向で変化率を計算せよ。ドル／円は値が小さいので、小数第6位まで表示する。
2. 二つの変化率は、符号を反転しただけの数値になるか。分母に使った値に注目して説明せよ。
3. 円／ドル表示の変化率を $r=S_1/S_0-1$ とすると、ドル／円表示の変化率が $-r/(1+r)$ になることを式で確かめよ。

余裕があれば `for year in ["2023", "2024", "2025"]:` で各年の最初と最後のレートから、同じ2つの変化率を求める。

In [ ]:
# first と last の逆数を計算し、表示方向ごとの変化率を求める。
# 余裕があれば for 文で2023〜2025年を比較する。


## 保存

コードを変えたら、後ろのセルも実行し直す。記入欄はダブルクリックで編集し、`Shift + Enter` で表示に戻す。答えには日付・金額・単位を書く。

Notebookの保存先はこの端末のブラウザの中である。終わったらダウンロードしておく。

## 出典

[日本銀行「外国為替市況」](https://www.boj.or.jp/statistics/market/forex/fxdaily/index.htm)の東京市場ドル・円スポット、17時時点（円／ドル）を使用。配布データの対象期間は2000〜2025年である。
